In [1]:
// load the formatting setup, the visualization functions, and the style sheet
#load "../setup.fsx"
#load "../visuals.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# The Eight Queens Puzzle

The <a href="https://en.wikipedia.org/wiki/Eight_queens_puzzle">eight queens puzzle</a> is the problem of placing eight chess queens on a chessboard so that no two queens can attack each other.  In <a href="https://en.wikipedia.org/wiki/Chess">chess</a> a queen can attack another piece if this piece is either
<ol>
    <li>in the same row,</li>
    <li>in the same column, or</li>
    <li>in the same diagonal.</li>
</ol>
The image below shows a queen in row 3, column 4.  All the locations where a piece can be captured by this queen are marked with an arrow.

<img src="queen-captures.png">

We will solve this puzzle by coding it as a formula of propositional logic.  This formula will be solvable iff the eight queens puzzle has a solution.  We will use the algorithm of *Davis and Putnam* to compute the solution of this formula.  We import the implementation that uses the Jeroslow-Wang heuristic.  This notebook also imports the types `Literal`, `Clause`, and `CNF` from the notebook `04-CNF.ipynb`.

In [2]:
importNotebook "07-Davis-Putnam-JW.ipynb"

imported 

Propositional-Logic-Parser.ipynb, 04-CNF.ipynb, 07-Davis-Putnam-JW.ipynb

The function `var` takes two arguments:
- `row` is the number of a row of the board,
- `col` is the number of a column of the board.

It returns the string `Q<row,col>`.  This string is interpreted as a propositional variable.  This variable is `true` iff there is a queen in the given row and column on the board.  The image below shows how theses variables correspond to the positions on a chess board.

<img src="queens-vars.png">

In [3]:
let var (row: int) (col: int) : string =
    sprintf "Q<%d,%d>" row col

In [4]:
var 11 3

"Q<11,3>"

The function `atMostOne` takes one argument:
- `S` is a set of propositional variables.

It returns a set of clauses that expresses the fact that **at most one** of the variables in $S$ is `true`.  For every pair of different variables $p$ and $q$ from $S$, the clause $\{\neg p, \neg q\}$ states that $p$ and $q$ are not both `true`.  The condition `p < q` ensures that every pair is only considered once.

In [5]:
let atMostOne (S: Set<string>) : CNF =
    set [ for p in S do
            for q in S do
                if p < q then
                    set [ Neg p; Neg q ] ]

In [6]:
atMostOne (set [ "a"; "b"; "c" ])

set [set [Neg "a"; Neg "b"]; set [Neg "a"; Neg "c"]; set [Neg "b"; Neg "c"]]

The function `atMostOneInRow` takes two arguments:
- `row` is the number of a row,
- `n` is the size of the board.

It returns a set of clauses that is `true` if and only if there is at most one queen in the given row.

In [7]:
let atMostOneInRow (row: int) (n: int) : CNF =
    let VarsInRow = set [ for col in 1 .. n -> var row col ]
    atMostOne VarsInRow

In [8]:
atMostOneInRow 3 4

set [set [Neg "Q<3,1>"; Neg "Q<3,2>"]; set [Neg "Q<3,1>"; Neg "Q<3,3>"]; set [Neg "Q<3,1>"; Neg "Q<3,4>"]; set [Neg "Q<3,2>"; Neg "Q<3,3>"]; set [Neg "Q<3,2>"; Neg "Q<3,4>"]; set [Neg "Q<3,3>"; Neg "Q<3,4>"]]

The function `oneInColumn` takes two arguments:
- `col` is the number of a column,
- `n` is the size of the board.

It returns a set of clauses that is `true` if and only if there is at least one queen in the given column.  This set contains just one clause: the disjunction of all variables of the given column.

In [9]:
let oneInColumn (col: int) (n: int) : CNF =
    let VarsInColumn = set [ for row in 1 .. n -> Pos (var row col) ]
    set [ VarsInColumn ]

In [10]:
oneInColumn 2 4

set [set [Pos "Q<1,2>"; Pos "Q<2,2>"; Pos "Q<3,2>"; Pos "Q<4,2>"]]

The function `atMostOneInFallingDiagonal` takes two arguments:
- `k` is a number specifying a falling diagonal,
- `n` is the size of the board.

It returns a set of clauses that is `true` if and only if there is at most one queen in the falling diagonal specified by the equation
```
     row - col = k.
```

In [11]:
let atMostOneInFallingDiagonal (k: int) (n: int) : CNF =
    let VarsInDiagonal = set [ for row in 1 .. n do
                                 for col in 1 .. n do
                                   if row - col = k then
                                     var row col ]
    atMostOne VarsInDiagonal

In [12]:
atMostOneInFallingDiagonal 0 4

set [set [Neg "Q<1,1>"; Neg "Q<2,2>"]; set [Neg "Q<1,1>"; Neg "Q<3,3>"]; set [Neg "Q<1,1>"; Neg "Q<4,4>"]; set [Neg "Q<2,2>"; Neg "Q<3,3>"]; set [Neg "Q<2,2>"; Neg "Q<4,4>"]; set [Neg "Q<3,3>"; Neg "Q<4,4>"]]

The function `atMostOneInRisingDiagonal` takes two arguments:
- `k` is a number specifying a rising diagonal,
- `n` is the size of the board.

It returns a set of clauses that is `true` if and only if there is at most one queen in the rising diagonal specified by the equation
```
    row + col = k. 
```

In [13]:
let atMostOneInRisingDiagonal (k: int) (n: int) : CNF =
    let VarsInDiagonal = set [ for row in 1 .. n do
                                 for col in 1 .. n do
                                   if row + col = k then
                                     var row col ]
    atMostOne VarsInDiagonal

In [14]:
atMostOneInRisingDiagonal 5 4

set [set [Neg "Q<1,4>"; Neg "Q<2,3>"]; set [Neg "Q<1,4>"; Neg "Q<3,2>"]; set [Neg "Q<1,4>"; Neg "Q<4,1>"]; set [Neg "Q<2,3>"; Neg "Q<3,2>"]; set [Neg "Q<2,3>"; Neg "Q<4,1>"]; set [Neg "Q<3,2>"; Neg "Q<4,1>"]]

The function `allClauses` takes one argument:
- `n` is the size of the board.

It returns a set of clauses that specify that
* there is at most one queen in every row,
* there is at most one queen in every rising diagonal,
* there is at most one queen in every falling diagonal, and
* there is at least one queen in every column.

**Implementation:** `All` is a list of sets of clauses.  `Set.unionMany` computes the union of all these sets.

In [15]:
let allClauses (n: int) : CNF =
    let All = [ for row in 1 .. n             -> atMostOneInRow row n           ]
            @ [ for k in 3 .. 2*n - 1         -> atMostOneInRisingDiagonal k n  ]
            @ [ for k in -(n - 2) .. (n - 2)  -> atMostOneInFallingDiagonal k n ]
            @ [ for col in 1 .. n             -> oneInColumn col n              ]
    Set.unionMany All

In [16]:
for C in allClauses 4 do
    printfn "%s" (clauseToString C)

{Q<1,1>, Q<2,1>, Q<3,1>, Q<4,1>}

{Q<1,2>, Q<2,2>, Q<3,2>, Q<4,2>}

{Q<1,3>, Q<2,3>, Q<3,3>, Q<4,3>}

{Q<1,4>, Q<2,4>, Q<3,4>, Q<4,4>}

{¬Q<1,1>, ¬Q<1,2>}

...

{¬Q<4,1>, ¬Q<4,3>}

{¬Q<4,1>, ¬Q<4,4>}

{¬Q<4,2>, ¬Q<4,3>}

{¬Q<4,2>, ¬Q<4,4>}

{¬Q<4,3>, ¬Q<4,4>}

For the 16 queens problem, there are $16 \cdot 16 = 256$ variables.  The number of clauses is computed as follows:

In [17]:
(allClauses 16).Count

4416

The function `queens` takes one argument:
- `n` is the size of the board.

It solves the *n queens problem*.  The result has the type `CNF option`:
- If the problem is solvable, `Some Solution` is returned, where `Solution` is a set of unit clauses that specifies the values of all variables.
- Otherwise, a message is printed and `None` is returned.

In [18]:
let queens (n: int) : CNF option =
    let Clauses  = allClauses n
    let Solution = solve Clauses
    if Solution <> set [ Set.empty ] then
        Some Solution
    else
        printfn "The problem is not solvable for %d queens!" n
        None

The *8 queens problem* can be solved quickly using the pure *Davis Putnam algorithm* that does not use the Jeroslow-Wang heuristic. If we want to solve the *16 queens problem*, we need to use the Jeroslow-Wang heuristic.

In [19]:
#!time
let Solution = queens 16

Wall time: 2941.9366ms

In [20]:
Solution

Some set [set [Pos "Q<1,2>"]; set [Pos "Q<10,1>"]; set [Pos "Q<11,4>"]; set [Pos "Q<12,9>"]; set [Pos "Q<13,11>"]; set [Pos "Q<14,14>"]; set [Pos "Q<15,3>"]; set [Pos "Q<16,10>"]; set [Pos "Q<2,8>"]; set [Pos "Q<3,6>"]; set [Pos "Q<4,13>"]; set [Pos "Q<5,7>"]; set [Pos "Q<6,16>"]; set [Pos "Q<7,12>"]; set [Pos "Q<8,15>"]; set [Pos "Q<9,5>"]; set [Neg "Q<1,10>"]; set [Neg "Q<1,11>"]; set [Neg "Q<1,12>"]; set [Neg "Q<1,13>"]; set [Neg "Q<1,14>"]; set [Neg "Q<1,15>"]; set [Neg "Q<1,16>"]; set [Neg "Q<1,1>"]; set [Neg "Q<1,3>"]; set [Neg "Q<1,4>"]; set [Neg "Q<1,5>"]; set [Neg "Q<1,6>"]; set [Neg "Q<1,7>"]; set [Neg "Q<1,8>"]; set [Neg "Q<1,9>"]; set [Neg "Q<10,10>"]; set [Neg "Q<10,11>"]; set [Neg "Q<10,12>"]; set [Neg "Q<10,13>"]; set [Neg "Q<10,14>"]; set [Neg "Q<10,15>"]; set [Neg "Q<10,16>"]; set [Neg "Q<10,2>"]; set [Neg "Q<10,3>"]; set [Neg "Q<10,4>"]; set [Neg "Q<10,5>"]; set [Neg "Q<10,6>"]; set [Neg "Q<10,7>"]; set [Neg "Q<10,8>"]; set [Neg "Q<10,9>"]; set [Neg "Q<11,10>"]; set [

The 3 queens problem has no solution:

In [21]:
queens 3

The problem is not solvable for 

3

 queens!

None

# Visualization

The function `extractRowCol` takes one argument:
- `name` is a string of the form `Q<row,col>`.

It returns the pair `(row, col)`.

**Implementation:** We remove the first two characters `Q<` and the last character `>`.  The remaining string `row,col` is split at the comma and both parts are converted to integers.

In [22]:
let extractRowCol (name: string) : int * int =
    let parts = name.Substring(2, name.Length - 3).Split(',')
    (int parts[0], int parts[1])

In [23]:
extractRowCol "Q<13,9>"

(13, 9)

The function `queenPositions` takes one argument:
- `Solution` is a set of unit clauses representing a solution of the n queens problem.

It returns the list of the positions `(row, col)` of all queens.  These positions correspond to the positive literals in `Solution`.

In [24]:
let queenPositions (Solution: CNF) : (int * int) list =
    [ for C in Solution do
        for l in C do
            match l with
            | Pos p -> extractRowCol p
            | Neg _ -> () ]

The function `showSolution` takes two arguments:
- `Solution` is a set of unit clauses representing a solution of the n queens problem,
- `width` specifies the size of the board as a CSS length, e.g. a percentage of the width of notebook.

It displays this solution on a chess board.  The function `chessBoard` is defined in the file `visuals.fsx`, which has been loaded in the first cell of this notebook.

In [25]:
let showSolution (Solution: CNF) (width: string) =
    let positions = queenPositions Solution
    chessBoard (List.length positions) positions width

In [26]:
showSolution Solution.Value "50%"

,♛,,,,,,,,,,,,,,
,,,,,,,♛,,,,,,,,
,,,,,♛,,,,,,,,,,
,,,,,,,,,,,,♛,,,
,,,,,,♛,,,,,,,,,
,,,,,,,,,,,,,,,♛
,,,,,,,,,,,♛,,,,
,,,,,,,,,,,,,,♛,
,,,,♛,,,,,,,,,,,
♛,,,,,,,,,,,,,,,
,,,♛,,,,,,,,,,,,
